In [5]:
import sys
print(sys.executable)


c:\Users\isabe\anaconda3\python.exe


In [2]:
!pip install psycopg2-binary


The %sql command uses SQLAlchemy to connect the database, but it needs a separate driver to communicate with PostgreSQL, which is psycopg2. 

In [ ]:
!pip install jupysql

In [2]:
%load_ext sql
%sql postgresql://postgres@localhost:5432/northwind


Connecting to 'postgresql://postgres@localhost:5432/northwind'

In [7]:
%config SqlMagic.displaylimit = 0 

In [12]:
%sql SELECT * FROM customers LIMIT 5;


Running query in 'postgresql://postgres@localhost:5432/northwind'

5 rows affected.

customer_id,company_name,contact_name,contact_title,address,city,region,postal_code,country,phone,fax
ALFKI,Alfreds Futterkiste,Maria Anders,Sales Representative,Obere Str. 57,Berlin,None,12209,Germany,030-0074321,030-0076545
ANATR,Ana Trujillo Emparedados y helados,Ana Trujillo,Owner,Avda. de la Constitución 2222,México D.F.,None,05021,Mexico,(5) 555-4729,(5) 555-3745
ANTON,Antonio Moreno Taquería,Antonio Moreno,Owner,Mataderos 2312,México D.F.,None,05023,Mexico,(5) 555-3932,None
AROUT,Around the Horn,Thomas Hardy,Sales Representative,120 Hanover Sq.,London,None,WA1 1DP,UK,(171) 555-7788,(171) 555-6750
BERGS,Berglunds snabbköp,Christina Berglund,Order Administrator,Berguvsvägen 8,Luleå,None,S-958 22,Sweden,0921-12 34 65,0921-12 34 67


In [13]:
%%sql
SELECT table_name AS name,
       table_type AS type
  FROM information_schema.tables
 WHERE table_schema = 'public' AND table_type IN ('BASE TABLE', 'VIEW');

Running query in 'postgresql://postgres@localhost:5432/northwind'

14 rows affected.

name,type
territories,BASE TABLE
order_details,BASE TABLE
employee_territories,BASE TABLE
us_states,BASE TABLE
customers,BASE TABLE
orders,BASE TABLE
employees,BASE TABLE
shippers,BASE TABLE
products,BASE TABLE
categories,BASE TABLE


In [15]:
%%sql
ALTER TABLE employees
DROP COLUMN photo;

Running query in 'postgresql://postgres@localhost:5432/northwind'

++
||
++
++

In [ ]:
# %%sql
# DROP VIEW IF EXISTS detailed_order;

Running query in 'postgresql://postgres@localhost:5432/northwind'

++
||
++
++

### Combine Customers and Orders

I joined the `customers` and `orders` tables on `customer_id` and saved the result as the `detailed_order` view. This combines customer details (company, contact, country, city, region) with order details (order date, shipped date, ship destination, employee) into a single view, so future analysis won't require repeating this join.

In [16]:
%%sql
CREATE OR REPLACE VIEW detailed_order AS
       SELECT c.customer_id,
       c.country,
       c.city,
       c.region,
       c.company_name,
       c.contact_name,
       o.order_id,
       o.order_date,
       o.shipped_date,
       o.ship_country,
       o.ship_city,
       o.employee_id
         FROM customers as c
         JOIN orders as o
           ON c.customer_id=o.customer_id;
SELECT * FROM detailed_order
LIMIT 10;


Running query in 'postgresql://postgres@localhost:5432/northwind'

10 rows affected.

customer_id,country,city,region,company_name,contact_name,order_id,order_date,shipped_date,ship_country,ship_city,employee_id
VINET,France,Reims,None,Vins et alcools Chevalier,Paul Henriot,10248,1996-07-04,1996-07-16,France,Reims,5
TOMSP,Germany,Münster,None,Toms Spezialitäten,Karin Josephs,10249,1996-07-05,1996-07-10,Germany,Münster,6
HANAR,Brazil,Rio de Janeiro,RJ,Hanari Carnes,Mario Pontes,10250,1996-07-08,1996-07-12,Brazil,Rio de Janeiro,4
VICTE,France,Lyon,None,Victuailles en stock,Mary Saveley,10251,1996-07-08,1996-07-15,France,Lyon,3
SUPRD,Belgium,Charleroi,None,Suprêmes délices,Pascale Cartrain,10252,1996-07-09,1996-07-11,Belgium,Charleroi,4
HANAR,Brazil,Rio de Janeiro,RJ,Hanari Carnes,Mario Pontes,10253,1996-07-10,1996-07-16,Brazil,Rio de Janeiro,3
CHOPS,Switzerland,Bern,None,Chop-suey Chinese,Yang Wang,10254,1996-07-11,1996-07-23,Switzerland,Bern,5
RICSU,Switzerland,Genève,None,Richter Supermarkt,Michael Holz,10255,1996-07-12,1996-07-15,Switzerland,Genève,9
WELLI,Brazil,Resende,SP,Wellington Importadora,Paula Parente,10256,1996-07-15,1996-07-17,Brazil,Resende,3
HILAA,Venezuela,San Cristóbal,Táchira,HILARION-Abastos,Carlos Hernández,10257,1996-07-16,1996-07-22,Venezuela,San Cristóbal,4


### Add Product Details to Orders

I joined `order_details`, `products`, and `orders` (via the `detailed_order` view) to bring in the product name and quantity for each order line, saving the result as the `detailed_order_info` view. This gives a complete, line-item-level view of what was ordered, so future analysis won't require repeating this join.

In [17]:
%%sql
CREATE OR REPLACE VIEW detailed_order_info as 
      SELECT od.order_id, 
             od.product_id,
             od.quantity,
             p.product_name
        FROM detailed_order as de
        JOIN employees as e
          ON e.employee_id=de.employee_id
        JOIN order_details as od
          ON de.order_id=od.order_id
        JOIN products as p
          ON p.product_id=od.product_id;
       
SELECT * FROM detailed_order_info
LIMIT 10

Running query in 'postgresql://postgres@localhost:5432/northwind'

10 rows affected.

order_id,product_id,quantity,product_name
10248,11,12,Queso Cabrales
10248,42,10,Singaporean Hokkien Fried Mee
10248,72,5,Mozzarella di Giovanni
10249,14,9,Tofu
10249,51,40,Manjimup Dried Apples
10250,41,10,Jack's New England Clam Chowder
10250,51,35,Manjimup Dried Apples
10250,65,15,Louisiana Fiery Hot Pepper Sauce
10251,22,6,Gustaf's Knäckebröd
10251,57,15,Ravioli Angelo


### Combine Orders with Employees

I joined `orders` with `customers` and `employees`, saving the result as the `orders_with_employees` view. I used a `LEFT JOIN` on `employees` (instead of an inner join) so that every order is kept even if its `employee_id` doesn't match a valid employee — for example due to a system error or missing data. This way, orders with no matching employee still show up with `NULL` employee fields instead of being silently dropped, making it easier to spot and investigate that kind of data issue.

In [ ]:
%%sql
CREATE OR REPLACE VIEW orders_with_employees AS
SELECT 
    o.order_id,
    c.customer_id, 
    c.contact_name, 
    e.employee_id,
    e.first_name || ' ' || e.last_name AS employee_full_name,
    e.title

FROM orders AS o  
JOIN customers AS c
    ON o.customer_id = c.customer_id
LEFT JOIN employees AS e
    ON o.employee_id = e.employee_id;

SELECT * from orders_with_employees
LIMIT 10

Running query in 'postgresql://postgres@localhost:5432/northwind'

10 rows affected.

order_id,customer_id,contact_name,employee_id,employee_full_name,title
10248,VINET,Paul Henriot,5,Steven Buchanan,Sales Manager
10249,TOMSP,Karin Josephs,6,Michael Suyama,Sales Representative
10250,HANAR,Mario Pontes,4,Margaret Peacock,Sales Representative
10251,VICTE,Mary Saveley,3,Janet Leverling,Sales Representative
10252,SUPRD,Pascale Cartrain,4,Margaret Peacock,Sales Representative
10253,HANAR,Mario Pontes,3,Janet Leverling,Sales Representative
10254,CHOPS,Yang Wang,5,Steven Buchanan,Sales Manager
10255,RICSU,Michael Holz,9,Anne Dodsworth,Sales Representative
10256,WELLI,Paula Parente,3,Janet Leverling,Sales Representative
10257,HILAA,Carlos Hernández,4,Margaret Peacock,Sales Representative


### Rank Employees by Total Sales

I used a CTE (`rank_employees`) to calculate each employee's total sales, joining `employees` to `orders` and `order_details` (with `LEFT JOIN`s so employees with no orders still appear) and summing `unit_price * quantity * (1 - discount)` per employee. Within the same CTE, I applied `RANK()` with an `OVER (ORDER BY ... DESC)` clause to assign each employee a sales rank, with ties receiving the same rank. The main query simply selects the results, giving each employee's total sales alongside their rank.

In [11]:
%%sql
WITH rank_employees AS (
       SELECT e.employee_id,
              e.first_name || ' ' || e.last_name AS full_name,
              e.title,
              ROUND(SUM(od.unit_price * od.quantity * (1 - od.discount))::NUMERIC, 2) AS total_sales,
              RANK() OVER (ORDER BY SUM(od.unit_price * od.quantity * (1 - od.discount)) DESC) AS sales_rank
         FROM employees as e
    LEFT JOIN orders as o
           ON o.employee_id = e.employee_id
    LEFT JOIN order_details as od
           ON od.order_id = o.order_id
        GROUP BY e.employee_id, full_name, title
)
SELECT *
  FROM rank_employees;

Running query in 'postgresql://postgres@localhost:5432/northwind'

9 rows affected.

employee_id,full_name,title,total_sales,sales_rank
4,Margaret Peacock,Sales Representative,232890.85,1
3,Janet Leverling,Sales Representative,202812.84,2
1,Nancy Davolio,Sales Representative,192107.60,3
2,Andrew Fuller,"Vice President, Sales",166537.76,4
8,Laura Callahan,Inside Sales Coordinator,126862.28,5
7,Robert King,Sales Representative,124568.23,6
9,Anne Dodsworth,Sales Representative,77308.07,7
6,Michael Suyama,Sales Representative,73913.13,8
5,Steven Buchanan,Sales Manager,68792.28,9


**Analysis:** **Sales Representatives take up most of the top ranks** in this table. That's not too surprising on its own, since Sales Representative is likely the most common title at the company — with more people in that group, it's more likely that some of them will land near the top just by having more employees in the pool. To fairly compare *titles* rather than individuals, **the next step would be to group by title, so each title is represented once**, instead of letting the title with the most employees dominate the individual-level ranking.

### Rank Employees by Total Sales Within Each Title

I extended the previous ranking query by adding `PARTITION BY e.title` to the `RANK()` window function. Instead of ranking all employees against each other, this resets the ranking within each job title group, so `RANK()` now shows how each employee's total sales compares only to peers who share the same `title` (e.g., ranking Sales Representatives against other Sales Representatives, separately from Sales Managers).

In [10]:
%%sql
WITH rank_employees AS (
       SELECT e.employee_id,
              e.first_name || ' ' || e.last_name AS full_name,
              e.title,
              ROUND(SUM(od.unit_price * od.quantity * (1 - od.discount))::NUMERIC, 2) AS total_sales,
              RANK() OVER (PARTITION BY e.title ORDER BY SUM(od.unit_price * od.quantity * (1 - od.discount)) DESC) AS sales_rank
         FROM employees as e
    LEFT JOIN orders as o
           ON o.employee_id = e.employee_id
    LEFT JOIN order_details as od
           ON od.order_id = o.order_id
        GROUP BY e.employee_id, full_name, title
)
SELECT *
  FROM rank_employees;

Running query in 'postgresql://postgres@localhost:5432/northwind'

9 rows affected.

employee_id,full_name,title,total_sales,sales_rank
8,Laura Callahan,Inside Sales Coordinator,126862.28,1
5,Steven Buchanan,Sales Manager,68792.28,1
4,Margaret Peacock,Sales Representative,232890.85,1
3,Janet Leverling,Sales Representative,202812.84,2
1,Nancy Davolio,Sales Representative,192107.60,3
7,Robert King,Sales Representative,124568.23,4
9,Anne Dodsworth,Sales Representative,77308.07,5
6,Michael Suyama,Sales Representative,73913.13,6
2,Andrew Fuller,"Vice President, Sales",166537.76,1


In [23]:
%%sql
SELECT title, COUNT(title)
FROM employees
GROUP BY title


Running query in 'postgresql://postgres@localhost:5432/northwind'

4 rows affected.

title,count
"Vice President, Sales",1
Inside Sales Coordinator,1
Sales Representative,6
Sales Manager,1


**Analysis:** This is a case where `PARTITION BY title` needs to be read carefully rather than taken at face value. Checking `employees`, all `4` titles at the company are sales-related (`Sales Representative`, `Sales Manager`, `Vice President, Sales`, `Inside Sales Coordinator`), but **only `Sales Representative` actually has more than one person — `6` employees — while the other three titles have exactly one employee each**. So for those single-employee titles, "rank 1 within title" isn't really a performance signal; it's just an artifact of there being no one else to rank against. **The only title where the within-group rank is comparing real peers is `Sales Representative`.**

### Monthly Sales with Running Total

I used a CTE (`joined_table`) to calculate total sales per month, joining `orders` to `order_details` and summing `unit_price * quantity * (1 - discount)`, grouped by month via `DATE_TRUNC('month', order_date)`. The main query then applies `SUM(total_sales_per_month) OVER (ORDER BY Month)` to produce a running total, so each row shows both that month's sales and the cumulative sales up to and including that month.

In [ ]:
%%sql
WITH joined_table AS (
    SELECT DATE_TRUNC('month', order_date) as Month,                            
        SUM(od.unit_price*od.quantity*(1-od.discount)) as total_sales_per_month
        FROM orders as o
        JOIN order_details as od
        ON o.order_id=od.order_id
       GROUP BY Month
)

SELECT *, SUM(total_sales_per_month) 
          OVER(order by Month) as running_totals
  FROM joined_table
 ORDER BY Month

Running query in 'postgresql://postgres@localhost:5432/northwind'

23 rows affected.

month,total_sales_per_month,running_totals
1996-07-01 00:00:00+00:00,27861.89512966156,27861.89512966156
1996-08-01 00:00:00+00:00,25485.275070743264,53347.17020040483
1996-09-01 00:00:00+00:00,26381.400132587554,79728.57033299239
1996-10-01 00:00:00+00:00,37515.72494547888,117244.29527847127
1996-11-01 00:00:00+00:00,45600.04521113701,162844.3404896083
1996-12-01 00:00:00+00:00,45239.630493214434,208083.97098282274
1997-01-01 00:00:00+00:00,61258.0701679784,269342.0411508011
1997-02-01 00:00:00+00:00,38483.6349503243,307825.6761011254
1997-03-01 00:00:00+00:00,38547.22010972678,346372.8962108522
1997-04-01 00:00:00+00:00,53032.95238894149,399405.8485997937


**Analysis:** The running total confirms Northwind's revenue grew steadily over the two years, but the month-to-month figures show it wasn't a smooth climb. Early on (1996-07 to 1996-12), monthly sales stayed in a tight `$26K–$45K` band. 1997 was more volatile — January spiked to `$61K`, then dropped in February. Between 1997-07 and 1997-12 sales trend upward overall, but with dips in `1997-08` and `1997-11`, before climbing steadily from `1998-01` through `1998-04`. There may be a promotion or seasonal event behind these spikes worth investigating, so the same strategy could be reused for future marketing. **One exception: `1998-05` shows a sharp drop, but that's because the dataset's order records may cut off mid-month, not a real sales decline — it should be excluded from this trend read.**

### Month-over-Month Growth Rate

I built on the monthly sales totals from the previous query, but instead of a running total, I used a CTE (`prev_month_sales`) with `LAG(total_sales_per_month) OVER (ORDER BY Month)` to pull each month's sales alongside the prior month's sales. The main query then calculates `growth_rate` as the percent change between the two — `(total_sales_per_month - prev_month_sales) / prev_month_sales * 100` — so each row shows how much that month's sales grew or shrank compared to the month before, rather than just the cumulative total.

In [26]:
%%sql
WITH total_sales_per_month AS (
     SELECT DATE_TRUNC('month', order_date) as Month,
            SUM(od.unit_price*od.quantity*(1-od.discount)) as current_month_sales
            FROM orders as o
            JOIN order_details as od
              ON o.order_id=od.order_id
           GROUP BY Month),
prev_month_sales AS (
    SELECT Month, current_month_sales, LAG(current_month_sales) OVER(
                             ORDER by Month) as prev_month_sales
      FROM total_sales_per_month
     GROUP BY Month, current_month_sales
)

SELECT Month, prev_month_sales, current_month_sales, ((current_month_sales-prev_month_sales)/prev_month_sales*100) as growth_rate
  FROM prev_month_sales 

Running query in 'postgresql://postgres@localhost:5432/northwind'

23 rows affected.

month,prev_month_sales,current_month_sales,growth_rate
1996-07-01 00:00:00+00:00,None,27861.89512966156,None
1996-08-01 00:00:00+00:00,27861.89512966156,25485.275070743264,-8.530001451294545
1996-09-01 00:00:00+00:00,25485.275070743264,26381.400132587554,3.51624637896504
1996-10-01 00:00:00+00:00,26381.400132587554,37515.72494547888,42.20520805162909
1996-11-01 00:00:00+00:00,37515.72494547888,45600.04521113701,21.54915112904513
1996-12-01 00:00:00+00:00,45600.04521113701,45239.630493214434,-0.7903823696967553
1997-01-01 00:00:00+00:00,45239.630493214434,61258.0701679784,35.40798079057388
1997-02-01 00:00:00+00:00,61258.0701679784,38483.6349503243,-37.17785290199861
1997-03-01 00:00:00+00:00,38483.6349503243,38547.22010972678,0.16522649038887202
1997-04-01 00:00:00+00:00,38547.22010972678,53032.95238894149,37.579187910257275


**Analysis:**

### Customer Order Frequency vs. Average

I used a CTE (`customer_order_values`) to count each customer's total orders and compare it against the overall average order count via `AVG(COUNT(...)) OVER()`. The main query then labels each customer as `'Above Average'` or `'Average/Below Average'` based on that comparison, making it easy to spot which customers order more or less frequently than the typical customer.

In [ ]:
%%sql
WITH customer_order_values AS (
    SELECT c.customer_id,
           c.contact_name, 
           c.country,
           COUNT(c.customer_id) as number_of_orders,
           ROUND(AVG(COUNT(c.customer_id)) OVER(),2) as average_order
    FROM orders as o
    JOIN customers as c
      ON c.customer_id=o.customer_id
   GROUP BY c.customer_id, c.contact_name, c.country
)

SELECT *,
CASE  
WHEN number_of_orders>average_order
THEN 'Above Average'
WHEN number_of_orders<average_order
THEN 'Average/Below Average'
END as order_category
FROM customer_order_values

### Customer Cohort Retention

Plan: group customers by the month of their first order (cohort), then for each cohort track what share of those customers placed another order in each following month. This separates whether revenue growth is coming from retaining existing customers or just from new customer acquisition each month.

In [ ]:
%%sql
